# Inference Demo — Intel Image Classification

Run predictions on single images or a folder using the best saved checkpoint.

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from omegaconf import OmegaConf
from pathlib import Path

from model import get_model
from dataset import _build_transforms

In [ ]:
# --- Config & device ---
cfg = OmegaConf.load('../configs/train_config.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# --- Load model ---
model = get_model(num_classes=cfg.num_classes).to(device)
ckpt = torch.load(f'../{cfg.checkpoint_dir}/{cfg.checkpoint_name}', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded checkpoint (val_acc={ckpt['val_acc']:.4f})")

In [ ]:
def predict(image_path: str) -> dict:
    """
    Run inference on a single image.
    
    Returns a dict with 'class', 'confidence', and 'top5'.
    """
    transform = _build_transforms('test')
    img = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1)[0].cpu()
    
    top5_probs, top5_ids = probs.topk(5)
    classes = list(cfg.classes)
    
    return {
        'class': classes[probs.argmax().item()],
        'confidence': probs.max().item(),
        'top5': [(classes[i], p.item()) for i, p in zip(top5_ids, top5_probs)],
        'image': img,
    }

In [ ]:
def show_prediction(image_path: str) -> None:
    """Display image alongside top-5 prediction bar chart."""
    result = predict(image_path)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    
    ax1.imshow(result['image'])
    ax1.set_title(f"Predicted: {result['class']}\n({result['confidence']*100:.1f}%)")
    ax1.axis('off')
    
    labels = [c for c, _ in result['top5']]
    values = [p * 100 for _, p in result['top5']]
    bars = ax2.barh(labels[::-1], values[::-1], color='steelblue')
    ax2.set_xlabel('Confidence (%)')
    ax2.set_title('Top-5 Predictions')
    ax2.set_xlim(0, 100)
    for bar, val in zip(bars, values[::-1]):
        ax2.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Run on a sample image ---
# Replace with an actual image path
image_path = '../data/intel-image-classification/test/buildings/20046.jpg'
show_prediction(image_path)

In [ ]:
# --- Batch inference on a folder ---
from torchvision import datasets
from torch.utils.data import DataLoader

test_dir = f'../{cfg.data_dir}/test'
transform = _build_transforms('test')
test_dataset = datasets.ImageFolder(test_dir, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=64, num_workers=4)

correct = 0
total = 0
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f'Test Accuracy: {correct/total:.4f} ({correct}/{total})')